# SEA Lesson Poll Generation Pipeline (Robust)

This notebook generates 1 poll per lesson JSON and inserts it as a `template_id: "poll"` segment.

Set the paths in the first cell, then run all.

In [1]:
# =========================================
# ENV LOADING (must be FIRST cell)
# =========================================

import os
from pathlib import Path

# Path to your .env file (adjust if needed)
ENV_PATH = Path(".env")

if ENV_PATH.exists():
    print(f"✅ Loading env from {ENV_PATH.resolve()}")
    for line in ENV_PATH.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip()
else:
    raise FileNotFoundError(f".env file not found at {ENV_PATH.resolve()}")

# Sanity check (prints only presence, not secrets)
print("ENV CHECK:")
for k in [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT",
    "AZURE_OPENAI_API_VERSION",
    "OPENAI_API_KEY",
]:
    print(f"  {k}: {'SET' if os.getenv(k) else 'NOT SET'}")


✅ Loading env from /Users/ben/Downloads/polls_pipeline_package/.env
ENV CHECK:
  AZURE_OPENAI_API_KEY: SET
  AZURE_OPENAI_ENDPOINT: SET
  AZURE_OPENAI_DEPLOYMENT: SET
  AZURE_OPENAI_API_VERSION: SET
  OPENAI_API_KEY: NOT SET


In [2]:
# --- CONFIG ---
# Point this at your English lesson folder (contains Module_* subfolders)
LESSONS_ROOT = "en"   # <-- change

# Where outputs should be written
OUTPUT_ROOT  = "polls"  # <-- change

TARGET_PER_LESSON = 1
MAX_ATTEMPTS = 4
SIMILARITY_THRESHOLD = 0.88
CRITIC_MIN_SCORE = 3
DEBUG_FAILURES = False
MAX_LESSONS = 0  # set >0 to run a small smoke test
OVERWRITE_EXISTING_POLLS = False

import os, sys, subprocess, json, textwrap, pathlib, re
print("LESSONS_ROOT:", LESSONS_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


LESSONS_ROOT: en
OUTPUT_ROOT: polls


In [3]:
# Quick sanity checks: ensure lesson files are discoverable before generating.

import os, re

LESSON_FILE_RE = re.compile(r"^(?P<m>\d+)\.(?P<c>\d+)\.(?P<l>-?\d+)\.json$")
def discover_lesson_files(root: str):
    paths=[]
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if not fn.endswith(".json"):
                continue
            m = LESSON_FILE_RE.match(fn)
            if not m:
                continue
            l = int(m.group("l"))
            if l < 1:
                continue
            paths.append(os.path.join(dirpath, fn))
    return sorted(paths)

lesson_files = discover_lesson_files(LESSONS_ROOT)
print("Lesson JSON files (lesson>=1) found:", len(lesson_files))
print("Sample:", lesson_files[:5])
if len(lesson_files) == 0:
    raise RuntimeError("No lesson files found. Check LESSONS_ROOT; expected files like 2.1.3.json within Module_* folders.")


Lesson JSON files (lesson>=1) found: 115
Sample: ['en/Module_1/1.1.1.json', 'en/Module_1/1.1.2.json', 'en/Module_1/1.1.3.json', 'en/Module_1/1.2.1.json', 'en/Module_1/1.2.2.json']


In [4]:
# Run the generator script (bundled alongside this notebook)

import os, subprocess, sys, pathlib, shlex

here = pathlib.Path().resolve()
script_path = here / "polls_pipeline.py"
if not script_path.exists():
    raise FileNotFoundError(f"polls_pipeline.py not found next to notebook: {script_path}")

cmd = [
    sys.executable, str(script_path),
    "--lessons_root", LESSONS_ROOT,
    "--output_root", OUTPUT_ROOT,
    "--target_per_lesson", str(TARGET_PER_LESSON),
    "--max_attempts", str(MAX_ATTEMPTS),
    "--similarity_threshold", str(SIMILARITY_THRESHOLD),
    "--critic_min_score", str(CRITIC_MIN_SCORE),
]
if OVERWRITE_EXISTING_POLLS:
    cmd.append("--overwrite_existing_polls")
if DEBUG_FAILURES:
    cmd.append("--debug_failures")
if MAX_LESSONS and int(MAX_LESSONS) > 0:
    cmd += ["--max_lessons", str(MAX_LESSONS)]

print("Running:", " ".join(shlex.quote(x) for x in cmd))

# Stream output live so progress logs appear immediately
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Generation failed with exit code {ret}.")


Running: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13 /Users/ben/Downloads/polls_pipeline_package/polls_pipeline.py --lessons_root en --output_root polls --target_per_lesson 1 --max_attempts 4 --similarity_threshold 0.88 --critic_min_score 3
✅ Found 115 lesson files under /Users/ben/Downloads/polls_pipeline_package/en
✅ Loaded registry entries: 309
▶️ [1/115] Processing 1.1.1 (Module_1/1.1.1.json)
  🔁 Attempt 1/4 generating poll...
  🧩 Archetype: policy_instrument
  ⏱️ LLM generate took 2.5s
  🔎 Running critic...
  ⏱️ Critic took 2.7s
  ✅ Generated poll 1.1.1__poll__3e4890db9da21b5c
▶️ [2/115] Processing 1.1.2 (Module_1/1.1.2.json)
  🔁 Attempt 1/4 generating poll...
  🧩 Archetype: tradeoff
  ⏱️ LLM generate took 2.2s
  🔎 Running critic...
  ⏱️ Critic took 2.5s
  ✅ Generated poll 1.1.2__poll__1eefed355a36b9eb
▶️ [3/115] Processing 1.1.3 (Module_1/1.1.3.json)
  🔁 Attempt 1/4 generating poll...
  🧩 Archetype: design_constraint
  ⏱️ LLM generate took 1.9s
  🔎 Running c

In [5]:
# Run the generator script (bundled alongside this notebook)

import os, subprocess, sys, pathlib, shlex

here = pathlib.Path().resolve()
script_path = here / "polls_pipeline.py"
if not script_path.exists():
    raise FileNotFoundError(f"polls_pipeline.py not found next to notebook: {script_path}")

cmd = [
    sys.executable, str(script_path),
    "--lessons_root", LESSONS_ROOT,
    "--output_root", OUTPUT_ROOT,
    "--target_per_lesson", str(TARGET_PER_LESSON),
    "--max_attempts", str(MAX_ATTEMPTS),
    "--similarity_threshold", str(SIMILARITY_THRESHOLD),
    "--critic_min_score", str(CRITIC_MIN_SCORE),
]
if OVERWRITE_EXISTING_POLLS:
    cmd.append("--overwrite_existing_polls")
if DEBUG_FAILURES:
    cmd.append("--debug_failures")
if MAX_LESSONS and int(MAX_LESSONS) > 0:
    cmd += ["--max_lessons", str(MAX_LESSONS)]

print("Running:", " ".join(shlex.quote(x) for x in cmd))

# Stream output live so progress logs appear immediately
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Generation failed with exit code {ret}.")


Running: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13 /Users/ben/Downloads/polls_pipeline_package/polls_pipeline.py --lessons_root en --output_root polls --target_per_lesson 1 --max_attempts 4 --similarity_threshold 0.88 --critic_min_score 3
✅ Found 115 lesson files under /Users/ben/Downloads/polls_pipeline_package/en
✅ Loaded registry entries: 412
▶️ [1/115] Processing 1.1.1 (Module_1/1.1.1.json)
  🔁 Attempt 1/4 generating poll...
  🧩 Archetype: failure_mode
  ⏱️ LLM generate took 1.9s
  🎭 Diversity gate failed (will still score candidate). (1 reasons)
  🔎 Running critic...
  ⏱️ Critic took 2.5s
  ⚠️ Candidate passed critic but failed diversity; retrying.
  🔁 Attempt 2/4 generating poll...
  🧩 Archetype: sequence_order
  🛠️ Using critic feedback (1 items) for retry
  ⏱️ LLM generate took 2.5s
  🎭 Diversity gate failed (will still score candidate). (1 reasons)
  🔎 Running critic...
  ⏱️ Critic took 2.4s
  ⚠️ Candidate passed critic but failed diversity; retrying.
 

In [6]:
# Run the generator script (bundled alongside this notebook)

import os, subprocess, sys, pathlib, shlex

here = pathlib.Path().resolve()
script_path = here / "polls_pipeline.py"
if not script_path.exists():
    raise FileNotFoundError(f"polls_pipeline.py not found next to notebook: {script_path}")

cmd = [
    sys.executable, str(script_path),
    "--lessons_root", LESSONS_ROOT,
    "--output_root", OUTPUT_ROOT,
    "--target_per_lesson", str(TARGET_PER_LESSON),
    "--max_attempts", str(MAX_ATTEMPTS),
    "--similarity_threshold", str(SIMILARITY_THRESHOLD),
    "--critic_min_score", str(CRITIC_MIN_SCORE),
]
if OVERWRITE_EXISTING_POLLS:
    cmd.append("--overwrite_existing_polls")
if DEBUG_FAILURES:
    cmd.append("--debug_failures")
if MAX_LESSONS and int(MAX_LESSONS) > 0:
    cmd += ["--max_lessons", str(MAX_LESSONS)]

print("Running:", " ".join(shlex.quote(x) for x in cmd))

# Stream output live so progress logs appear immediately
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Generation failed with exit code {ret}.")


Running: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13 /Users/ben/Downloads/polls_pipeline_package/polls_pipeline.py --lessons_root en --output_root polls --target_per_lesson 1 --max_attempts 4 --similarity_threshold 0.88 --critic_min_score 3
✅ Found 115 lesson files under /Users/ben/Downloads/polls_pipeline_package/en
✅ Loaded registry entries: 515
▶️ [1/115] Processing 1.1.1 (Module_1/1.1.1.json)
  🔁 Attempt 1/4 generating poll...
  🧩 Archetype: policy_instrument
  ⏱️ LLM generate took 2.0s
  🔎 Running critic...
  ⏱️ Critic took 2.6s
  ✅ Generated poll 1.1.1__poll__0ff57fccc1afc533
▶️ [2/115] Processing 1.1.2 (Module_1/1.1.2.json)
  🔁 Attempt 1/4 generating poll...
  🧩 Archetype: scenario_first_step
  ⏱️ LLM generate took 1.8s
  🔎 Running critic...
  ⏱️ Critic took 2.4s
  ✅ Generated poll 1.1.2__poll__c2c8fc644fef4f10
▶️ [3/115] Processing 1.1.3 (Module_1/1.1.3.json)
  🔁 Attempt 1/4 generating poll...
  🧩 Archetype: evidence_change_mind
  ⏱️ LLM generate took 2.0s